<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/3.2/30_train_predictions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 30_train_predictions.ipynb
# -------------------------------------------------------------------
# Purpose
# - Train classification (direction: prob_up) and regression (magnitude: pred_ret).
# - Export prediction files and model-level metrics only (no backtests).
# - Centralise analysis/app artefacts under 60_summary/.
#
# Notes
# - Australian English comments.
# - Minimal dependencies: pandas, numpy, scikit-learn, xgboost (optional).
# - Safe to run in Colab with Google Drive mounted to /content/drive.
# -------------------------------------------------------------------

# === 0) Setup & Paths ===
from pathlib import Path
import pandas as pd
import numpy as np
import datetime as dt
import json, os, sys, warnings, shutil
from hashlib import md5
warnings.filterwarnings("ignore")

# Mount Google Drive in Colab; skip if running locally.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Project root on Google Drive
PROJECT_DIR = Path("/content/drive/MyDrive/gt-markets")
DATA_DIR    = PROJECT_DIR / "data"
OUT_DIR     = PROJECT_DIR / "outputs"

# Short, meaningful run name (change as needed)
RUN_ID   = "3.1"
RUN_ROOT = OUT_DIR / "runs" / RUN_ID

# Create stage folders
for sub in ["00_meta", "10_logs", "40_figs", "50_leaderboards"]:
    (RUN_ROOT / sub).mkdir(parents=True, exist_ok=True)

# Per-frequency prediction folders
for f in ["D", "W"]:
    (RUN_ROOT / "20_preds" / f / "val").mkdir(parents=True, exist_ok=True)
    (RUN_ROOT / "20_preds" / f / "test").mkdir(parents=True, exist_ok=True)

# Centralised summary folder
(RUN_ROOT / "60_summary").mkdir(parents=True, exist_ok=True)

print("RUN_ROOT:", RUN_ROOT)

# === Debug toggle (use a very small slice for quick demos) ===
DEBUG = False         # Set to False for full dataset
DEBUG_ROWS = 220     # Keep the last N rows only in debug mode

if DEBUG:
    print(f"⚠️ DEBUG MODE: using last {DEBUG_ROWS} rows only")
else:
    print("✅ FULL RUN")

# === 1) Assets, Frequencies, Dataset variants ===
ASSET_CLOSE_COLS = {
    "BTC":    "BTC-USD Close",
    "GOLD":   "GC=F Close",
    "OIL":    "CL=F Close",
    "USDCNY": "USDCNY=X Close",
}
ASSETS = list(ASSET_CLOSE_COLS.keys())
FREQS  = ["D", "W"]  # Daily and Weekly

# Dataset variants and their source files (paths reflect prior pipeline outputs)
DATASET_TAGS = ["base", "eng", "ext"]

BASE_SRC = (DATA_DIR / "processed" / "merged_financial_trends_data_2025-09-07.csv")
ENG_SRC  = (DATA_DIR / "processed" / "merged_financial_trends_engineered_2025-09-07.csv")
KW_SRC   = (DATA_DIR / "Keyword Selection" / "combined_significant_lagged_correlations.csv")

# Cache original inputs inside the run for reproducibility
INPUT_CACHE_DIR = RUN_ROOT / "00_meta" / "input"
INPUT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def _md5sum(p: Path) -> str:
    h = md5()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def cached_copy(src: Path, cache_dir: Path) -> Path:
    """Copy src into cache if missing or changed; return cached path."""
    assert src.exists(), f"Missing required input: {src}"
    dst = cache_dir / src.name
    if not dst.exists():
        shutil.copy2(src, dst)
    else:
        try:
            if _md5sum(src) != _md5sum(dst):
                shutil.copy2(src, dst)
        except Exception:
            shutil.copy2(src, dst)
    return dst

# Copy and record inputs
BASE_CACHED = cached_copy(BASE_SRC, INPUT_CACHE_DIR)
ENG_CACHED  = None
KW_CACHED   = None

if "eng" in DATASET_TAGS:
    try:
        ENG_CACHED = cached_copy(ENG_SRC, INPUT_CACHE_DIR)
    except AssertionError as e:
        print("[warn] Engineered CSV not found; 'eng' dataset will be skipped:", e)

if "ext" in DATASET_TAGS:
    try:
        KW_CACHED = cached_copy(KW_SRC, INPUT_CACHE_DIR)
    except AssertionError as e:
        print("[warn] Keyword selection CSV not found; 'ext' dataset will use base features:", e)

# Update run metadata
meta_path = RUN_ROOT / "00_meta" / "run_meta.json"
try:
    with open(meta_path, "r") as f:
        run_meta = json.load(f)
except Exception:
    run_meta = {}

run_meta.update({
    "run_id": RUN_ID,
    "created_at": dt.datetime.now().isoformat(timespec="seconds"),
    "debug": bool(DEBUG),
    "debug_rows": int(DEBUG_ROWS),
    "datasets": DATASET_TAGS,
    "dataset_files": {
        "base": str(BASE_CACHED) if BASE_CACHED else None,
        "eng":  str(ENG_CACHED)  if ENG_CACHED  else None,
        "ext_keywords": str(KW_CACHED) if KW_CACHED else None,
    },
    "assets": ASSETS,
    "freqs": FREQS,
    "close_map": ASSET_CLOSE_COLS,
    "notebook_tag": "30_train_predictions",
})
with open(meta_path, "w") as f:
    json.dump(run_meta, f, indent=2)

# === 2) Data loading, resampling, labelling ===
def load_merged_df(path: Path) -> pd.DataFrame:
    """Load a merged table, enforce Date index, drop duplicates, apply debug truncation."""
    df = pd.read_csv(path)
    assert "Date" in df.columns, "Missing 'Date' column"
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
    df = df.loc[~df.index.duplicated(keep="last")]
    if DEBUG:
        df = df.tail(DEBUG_ROWS)
    return df

def to_frequency(df: pd.DataFrame, freq: str, close_cols: list[str]) -> pd.DataFrame:
    """
    Resample to the target frequency.
    - For Close columns, use last in period (financial convention).
    - For other numeric features, use mean in period.
    """
    close_res = df[close_cols].resample(freq).last()
    other_cols = [c for c in df.columns if c not in close_cols and pd.api.types.is_numeric_dtype(df[c])]
    if other_cols:
        feat_res = df[other_cols].resample(freq).mean()
        out = pd.concat([close_res, feat_res], axis=1)
    else:
        out = close_res
    out = out.dropna(how="all")
    return out

def add_targets_for_asset(df_freq: pd.DataFrame, close_col: str) -> pd.DataFrame:
    """
    Create next-period targets:
    - y_true_cls: 1 if next close > current close else 0.
    - y_true_ret: next-period return in percentage points.
    """
    df = df_freq.copy()
    px = df[close_col].ffill()
    df["y_true_ret"] = (px.pct_change().shift(-1)) * 100.0
    df["y_true_cls"] = (px.shift(-1) > px).astype(int)
    df = df.dropna(subset=["y_true_ret", "y_true_cls"])
    return df

# === 3) Feature selection per dataset flavour ===
def select_feature_columns(df: pd.DataFrame, close_col: str, dataset_tag: str, kw_file: Path | None) -> list[str]:
    """
    Select a feature set based on dataset flavour.
    - Always exclude the primary Close column and target columns.
    - 'base': all numeric features available (excluding Close and targets).
    - 'eng' : prefer engineered prefixes; fallback to numeric features if none found.
    - 'ext' : attempt to use keyword-selected columns; fallback to prefixed keyword columns; else numeric features.
    """
    drop_cols = {close_col, "y_true_ret", "y_true_cls"}
    numeric_cols = [c for c in df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])]

    if dataset_tag == "base":
        return numeric_cols

    if dataset_tag == "eng":
        engineered_prefixes = ("TA_", "ENG_", "FEAT_", "TECH_", "ROLL_")
        eng_cols = [c for c in numeric_cols if c.startswith(engineered_prefixes)]
        return eng_cols if len(eng_cols) > 0 else numeric_cols

    if dataset_tag == "ext":
        # Try to extract candidate feature names from the keyword selection CSV
        if kw_file is not None and Path(kw_file).exists():
            try:
                kw_df = pd.read_csv(kw_file)
                candidate_cols = [c for c in ["feature", "series", "name", "keyword", "column"] if c in kw_df.columns]
                kw_names = set()
                for c in candidate_cols:
                    kw_names.update(kw_df[c].astype(str).str.strip().tolist())
                # If lag values exist, consider lagged variants
                for c in ["lag", "lead", "shift"]:
                    if c in kw_df.columns:
                        lags = pd.to_numeric(kw_df[c], errors="coerce").dropna().astype(int).unique().tolist()
                        if len(lags) > 0:
                            kw_names.update({f"{name}_lag{n}" for name in list(kw_names) for n in lags})
                ext_cols = [c for c in numeric_cols if (c in kw_names)]
                if len(ext_cols) > 0:
                    return ext_cols
            except Exception as e:
                print("[warn] Keyword selection parsing failed; fallback to prefixed columns. Err:", e)

        # Fallback to common keyword/text prefixes if present in the merged table
        kw_prefixes = ("KW_", "NEWS_", "SENT_", "GTR_", "TW_", "RED_", "HEAD_")
        ext_cols = [c for c in numeric_cols if c.startswith(kw_prefixes)]
        return ext_cols if len(ext_cols) > 0 else numeric_cols

    return numeric_cols

def build_X_for_dataset(dfT: pd.DataFrame, close_col: str, dataset_tag: str) -> tuple[np.ndarray, list[str]]:
    """Construct feature matrix for the selected dataset flavour."""
    feat_cols = select_feature_columns(dfT, close_col, dataset_tag, KW_CACHED)
    X = dfT[feat_cols].fillna(0.0).values
    return X, feat_cols

# === 4) Adaptive walk-forward splits ===
def make_walk_forward_splits(index, n_folds=3, min_train=None, val_len=None):
    """
    Create time-ordered splits with adaptive defaults.
    Suitable for small debug slices and larger full runs.
    """
    n = len(index)
    if min_train is None:
        min_train = max(60, int(0.55 * n))
    if val_len is None:
        val_len   = max(12, int(0.20 * n))
    idx = np.arange(n)
    splits, start = [], min_train
    while start + val_len <= n and len(splits) < n_folds:
        splits.append((idx[:start], idx[start:start + val_len]))
        start += val_len
    return splits

# === 5) Models and metrics ===
try:
    from xgboost import XGBClassifier, XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False
    print("Note: XGBoost not available; proceeding without it.")

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

def get_classification_models():
    models = {
        "RF_cls": RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    }
    if HAS_XGB:
        models["XGB_cls"] = XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1,
            eval_metric="logloss",
        )
    return models

def get_regression_models():
    models = {
        "RF_reg": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    }
    if HAS_XGB:
        models["XGB_reg"] = XGBRegressor(
            n_estimators=500, max_depth=5, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, random_state=42, n_jobs=-1
        )
    return models

def eval_classification(y_true, prob_up):
    """Compute classification metrics with guards for small samples."""
    y_pred = (prob_up >= 0.5).astype(int)
    m = {"AUC": np.nan, "ACC": np.nan, "F1": np.nan}
    try:
        if len(np.unique(y_true)) > 1:
            m["AUC"] = roc_auc_score(y_true, prob_up)
        m["ACC"] = accuracy_score(y_true, y_pred)
        m["F1"]  = f1_score(y_true, y_pred, zero_division=0)
    except Exception:
        pass
    return m

def eval_regression(y_true, y_pred):
    """Compute regression metrics and directional hit."""
    out = {"MAE": np.nan, "RMSE": np.nan, "Spearman": np.nan, "DirHit": np.nan}
    try:
        out["MAE"]  = mean_absolute_error(y_true, y_pred)
        out["RMSE"] = mean_squared_error(y_true, y_pred, squared=False)
        rho, _ = spearmanr(y_true, y_pred)
        out["Spearman"] = rho
        out["DirHit"]   = float(np.mean(np.sign(y_true) == np.sign(y_pred)))
    except Exception:
        pass
    return out

# === 6) Training per asset × freq × dataset ===
def run_asset_freq_train_dataset(df_src, asset: str, freq: str, run_root: Path, dataset_tag: str):
    """
    Resample, label, build dataset-specific features, train models with walk-forward splits,
    write fold-level validation predictions and aggregated OOS predictions, and return metrics frames.
    """
    close_col = ASSET_CLOSE_COLS[asset]
    assert close_col in df_src.columns, f"Missing close column for {asset}: {close_col}"

    # Resample and label
    df_freq = to_frequency(df_src, freq, [close_col])
    dfT     = add_targets_for_asset(df_freq, close_col)

    # Features and labels
    X, feat_cols = build_X_for_dataset(dfT, close_col, dataset_tag)
    y_cls = dfT["y_true_cls"].values.astype(int)
    y_ret = dfT["y_true_ret"].values.astype(float)

    # Splits
    splits = make_walk_forward_splits(dfT.index, n_folds=(3 if freq == "D" else 2))
    if len(splits) == 0:
        print(f"[warn] Not enough data for splits: {asset} {freq} ({dataset_tag})")
        return [], []

    cls_models = get_classification_models()
    reg_models = get_regression_models()

    rows_leader = []
    oos_frames  = []

    val_dir  = run_root / "20_preds" / freq / "val"
    test_dir = run_root / "20_preds" / freq / "test"

    # Classification
    for model_name, model in cls_models.items():
        prob_up_all = np.full_like(y_cls, fill_value=np.nan, dtype=float)

        for k, (tr_idx, va_idx) in enumerate(splits, start=1):
            Xtr, ytr = X[tr_idx], y_cls[tr_idx]
            Xva, yva = X[va_idx], y_cls[va_idx]

            model.fit(Xtr, ytr)
            if hasattr(model, "predict_proba"):
                prob = model.predict_proba(Xva)[:, 1]
            elif hasattr(model, "decision_function"):
                scores = model.decision_function(Xva)
                prob = 1.0 / (1.0 + np.exp(-scores))
            else:
                prob = model.predict(Xva).astype(float)

            prob_up_all[va_idx] = prob

            pd.DataFrame({
                "date": dfT.index[va_idx],
                "asset": asset, "freq": freq, "dataset": dataset_tag,
                "model": model_name, "task": "classification",
                "window": "wf", "fold": k,
                "y_true_cls": yva, "y_true_ret": y_ret[va_idx],
                "prob_up": prob, "pred_ret": np.nan,
            }).to_csv(val_dir / f"{asset}_{freq}_{dataset_tag}_{model_name}_CLS_fold{k}.csv", index=False)

        mask = ~np.isnan(prob_up_all)
        cls_metrics = eval_classification(y_cls[mask], prob_up_all[mask])
        rows_leader.append({
            "asset": asset, "freq": freq, "dataset": dataset_tag,
            "model": model_name, "task": "classification",
            **cls_metrics, "n_samples_oos": int(mask.sum())
        })

        df_oos = pd.DataFrame({
            "date": dfT.index[mask],
            "asset": asset, "freq": freq, "dataset": dataset_tag,
            "model": model_name, "task": "classification",
            "window": "wf", "fold": -1,
            "y_true_cls": y_cls[mask], "y_true_ret": y_ret[mask],
            "prob_up": prob_up_all[mask], "pred_ret": np.nan,
        })
        df_oos.to_csv(test_dir / f"{asset}_{freq}_{dataset_tag}_{model_name}_CLS_OOS.csv", index=False)
        oos_frames.append(df_oos)

    # Regression
    for model_name, model in reg_models.items():
        yhat_all = np.full_like(y_ret, fill_value=np.nan, dtype=float)

        for k, (tr_idx, va_idx) in enumerate(splits, start=1):
            Xtr, ytr = X[tr_idx], y_ret[tr_idx]
            Xva, yva = X[va_idx], y_ret[va_idx]

            model.fit(Xtr, ytr)
            yhat = model.predict(Xva)
            yhat_all[va_idx] = yhat

            pd.DataFrame({
                "date": dfT.index[va_idx],
                "asset": asset, "freq": freq, "dataset": dataset_tag,
                "model": model_name, "task": "regression",
                "window": "wf", "fold": k,
                "y_true_cls": y_cls[va_idx], "y_true_ret": yva,
                "prob_up": np.nan, "pred_ret": yhat,
            }).to_csv(val_dir / f"{asset}_{freq}_{dataset_tag}_{model_name}_REG_fold{k}.csv", index=False)

        mask = ~np.isnan(yhat_all)
        reg_metrics = eval_regression(y_ret[mask], yhat_all[mask])
        rows_leader.append({
            "asset": asset, "freq": freq, "dataset": dataset_tag,
            "model": model_name, "task": "regression",
            **reg_metrics, "n_samples_oos": int(mask.sum())
        })

        df_oos = pd.DataFrame({
            "date": dfT.index[mask],
            "asset": asset, "freq": freq, "dataset": dataset_tag,
            "model": model_name, "task": "regression",
            "window": "wf", "fold": -1,
            "y_true_cls": y_cls[mask], "y_true_ret": y_ret[mask],
            "prob_up": np.nan, "pred_ret": yhat_all[mask],
        })
        df_oos.to_csv(test_dir / f"{asset}_{freq}_{dataset_tag}_{model_name}_REG_OOS.csv", index=False)
        oos_frames.append(df_oos)

    return rows_leader, oos_frames

# === 7) RUN (kept last) ===
print("Datasets:", DATASET_TAGS)

all_leader_rows = []
all_oos_frames  = []

for dataset_tag in DATASET_TAGS:
    # Skip engineered if the file is not available
    if dataset_tag == "eng" and not (ENG_CACHED and Path(ENG_CACHED).exists()):
        print("[skip] 'eng' requested but engineered CSV missing in cache.")
        continue

    # Load source for this dataset
    if dataset_tag == "base":
        df_src = load_merged_df(BASE_CACHED)
    elif dataset_tag == "eng":
        df_src = load_merged_df(ENG_CACHED)
    else:  # ext
        # Use engineered table if present, otherwise base; feature selection will extract keyword features if available
        df_src = load_merged_df(ENG_CACHED) if (ENG_CACHED and Path(ENG_CACHED).exists()) else load_merged_df(BASE_CACHED)

    print(f"→ Dataset [{dataset_tag}] shape: {df_src.shape} | {df_src.index.min()} → {df_src.index.max()}")

    for asset in ASSETS:
        for freq in FREQS:
            print(f"▶️  Training {asset} [{freq}] on dataset={dataset_tag} ...")
            try:
                rows_leader, oos_frames = run_asset_freq_train_dataset(df_src, asset, freq, RUN_ROOT, dataset_tag)
                all_leader_rows.extend(rows_leader)
                all_oos_frames.extend(oos_frames)
            except AssertionError as e:
                print("[warn]", e)
                continue

print("✅ Completed training and prediction export (no backtests).")

# === 8) Build 60_summary artefacts ===
summary_dir = RUN_ROOT / "60_summary"

# 8.1 predictions_master.csv (aggregated OOS across datasets)
if len(all_oos_frames) > 0:
    predictions_master = pd.concat(all_oos_frames, ignore_index=True).sort_values(["asset","freq","dataset","date"])
else:
    predictions_master = pd.DataFrame(columns=[
        "date","asset","freq","dataset","model","task","window","fold",
        "y_true_cls","y_true_ret","prob_up","pred_ret"
    ])
predictions_master.to_csv(summary_dir / "predictions_master.csv", index=False)

# 8.2 leaderboards_master.csv (model quality, no trading)
leaderboards_master = pd.DataFrame(all_leader_rows) if len(all_leader_rows) else pd.DataFrame(
    columns=["asset","freq","dataset","model","task","AUC","ACC","F1","MAE","RMSE","Spearman","DirHit","n_samples_oos"]
)
leaderboards_master["run_id"]     = RUN_ID
leaderboards_master["created_at"] = dt.datetime.now().isoformat(timespec="seconds")
leaderboards_master.to_csv(summary_dir / "leaderboards_master.csv", index=False)

# 8.3 predictions_coverage.csv (simple availability snapshot)
def build_coverage(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["asset","freq","dataset","model","n_pred","start_date","end_date","pct_missing"])
    g = df.groupby(["asset","freq","dataset","model"], as_index=False)
    rows = []
    for (a,f,d,m), sub in g:
        sub = sub.sort_values("date")
        n_pred = len(sub)
        start  = pd.to_datetime(sub["date"].iloc[0]).date() if n_pred else None
        end    = pd.to_datetime(sub["date"].iloc[-1]).date() if n_pred else None
        rows.append({"asset":a,"freq":f,"dataset":d,"model":m,
                     "n_pred":n_pred,"start_date":start,"end_date":end,"pct_missing":np.nan})
    return pd.DataFrame(rows)

coverage = build_coverage(predictions_master)
coverage.to_csv(summary_dir / "predictions_coverage.csv", index=False)

# 8.4 signals_latest.csv (latest OOS per asset×freq×dataset×model)
def build_latest_signals(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["date","asset","freq","dataset","model","prob_up","pred_ret","is_bigmove_pred","signal_strength"])
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])
    latest = df.sort_values("date").groupby(["asset","freq","dataset","model"], as_index=False).tail(1).copy()
    # Fixed big-move thresholds; can be refined later
    THRESH = {
        ("BTC","D"):0.04, ("BTC","W"):0.06,
        ("GOLD","D"):0.010, ("GOLD","W"):0.015,
        ("OIL","D"):0.02, ("OIL","W"):0.025,
        ("USDCNY","D"):0.002, ("USDCNY","W"):0.004,
    }
    latest["is_bigmove_pred"] = [
        int((abs((r or 0)/100.0) > THRESH.get((a,f), 0.02)) if (r==r) else False)
        for a,f,r in zip(latest["asset"], latest["freq"], latest["pred_ret"])
    ]
    latest["signal_strength"] = latest.groupby(["asset","freq","dataset"])["pred_ret"].apply(
        lambda s: (s.abs().rank(pct=True)).fillna(0.0)
    ).values
    return latest[["date","asset","freq","dataset","model","prob_up","pred_ret","is_bigmove_pred","signal_strength"]]

signals_latest = build_latest_signals(predictions_master)
signals_latest.to_csv(summary_dir / "signals_latest.csv", index=False)

# 8.5 data_dictionary.csv (concise schema reference)
data_dict_rows = [
    {"column":"date","dtype":"datetime","description":"Timestamp of observation (UTC).","example":"2025-09-05"},
    {"column":"asset","dtype":"str","description":"Asset code.","example":"BTC"},
    {"column":"freq","dtype":"str","description":"Sampling frequency (D=Daily, W=Weekly).","example":"D"},
    {"column":"dataset","dtype":"str","description":"Dataset variant (base, eng, ext).","example":"eng"},
    {"column":"model","dtype":"str","description":"Model identifier.","example":"XGB_reg"},
    {"column":"task","dtype":"str","description":"Task type (classification, regression).","example":"regression"},
    {"column":"window","dtype":"str","description":"CV/windowing label.","example":"wf"},
    {"column":"fold","dtype":"int","description":"Fold index (-1 denotes aggregated OOS).","example":"-1"},
    {"column":"y_true_cls","dtype":"int","description":"True direction label (1=up, 0=not up).","example":"1"},
    {"column":"y_true_ret","dtype":"float","description":"True next-period return (percentage points).","example":"1.25"},
    {"column":"prob_up","dtype":"float","description":"Predicted probability of an up-move (0..1).","example":"0.63"},
    {"column":"pred_ret","dtype":"float","description":"Predicted next-period return (percentage points).","example":"1.10"},
    {"column":"is_bigmove_pred","dtype":"int","description":"Derived flag; 1 if |pred_ret| exceeds the threshold for the asset×freq.","example":"0"},
    {"column":"signal_strength","dtype":"float","description":"Normalised signal strength (0..1) from |pred_ret| rank within group.","example":"0.82"},
]
pd.DataFrame(data_dict_rows).to_csv(summary_dir / "data_dictionary.csv", index=False)

print("📦 Summary files written to:", summary_dir)
print(" - predictions_master.csv")
print(" - leaderboards_master.csv")
print(" - predictions_coverage.csv")
print(" - signals_latest.csv")
print(" - data_dictionary.csv")
print("🗂  Inputs cached at:", INPUT_CACHE_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RUN_ROOT: /content/drive/MyDrive/gt-markets/outputs/runs/3.1
✅ FULL RUN
Datasets: ['base', 'eng', 'ext']
→ Dataset [base] shape: (2609, 170) | 2015-09-08 00:00:00 → 2025-09-05 00:00:00
▶️  Training BTC [D] on dataset=base ...
▶️  Training BTC [W] on dataset=base ...
▶️  Training GOLD [D] on dataset=base ...
▶️  Training GOLD [W] on dataset=base ...
▶️  Training OIL [D] on dataset=base ...
▶️  Training OIL [W] on dataset=base ...
▶️  Training USDCNY [D] on dataset=base ...
▶️  Training USDCNY [W] on dataset=base ...
→ Dataset [eng] shape: (2609, 254) | 2015-09-08 00:00:00 → 2025-09-05 00:00:00
▶️  Training BTC [D] on dataset=eng ...
▶️  Training BTC [W] on dataset=eng ...
▶️  Training GOLD [D] on dataset=eng ...
▶️  Training GOLD [W] on dataset=eng ...
▶️  Training OIL [D] on dataset=eng ...
▶️  Training OIL [W] on dataset=eng ...
▶️  Training USDCNY [D] on da

In [5]:
# === Add-on cell: Build run 3.2 with full model set (ML + DL) ===
import shutil

# New run id
RUN_ID_NEW   = "3.2"
RUN_ROOT_NEW = OUT_DIR / "runs" / RUN_ID_NEW

# Create folders
for sub in ["00_meta", "10_logs", "40_figs", "50_leaderboards"]:
    (RUN_ROOT_NEW / sub).mkdir(parents=True, exist_ok=True)
for f in ["D","W"]:
    (RUN_ROOT_NEW / "20_preds" / f / "val").mkdir(parents=True, exist_ok=True)
    (RUN_ROOT_NEW / "20_preds" / f / "test").mkdir(parents=True, exist_ok=True)
(RUN_ROOT_NEW / "60_summary").mkdir(parents=True, exist_ok=True)

print("RUN_ROOT_NEW:", RUN_ROOT_NEW)

# Copy cached inputs from 3.1 to 3.2
CACHE_SRC = OUT_DIR / "runs" / "3.1" / "00_meta" / "input"
CACHE_DST = RUN_ROOT_NEW / "00_meta" / "input"
CACHE_DST.mkdir(parents=True, exist_ok=True)
for f in CACHE_SRC.glob("*"):
    shutil.copy2(f, CACHE_DST)
print("Copied cached inputs from 3.1 → 3.2")

# Load 3.1 results
sum31 = OUT_DIR / "runs" / "3.1" / "60_summary"
preds_31 = pd.read_csv(sum31 / "predictions_master.csv")
leads_31 = pd.read_csv(sum31 / "leaderboards_master.csv")
print("Loaded 3.1 results:", preds_31.shape, leads_31.shape)

# === Define extra models (LR, GRU, LSTM, MLP) ===
from sklearn.linear_model import LogisticRegression, LinearRegression
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GRU, LSTM
from tensorflow.keras.optimizers import Adam

def get_extra_classification_models():
    return {
        "LR_cls": LogisticRegression(max_iter=500, n_jobs=-1),
        "MLP_cls": ("dl", "MLP_cls"),
        "GRU_cls": ("dl", "GRU_cls"),
        "LSTM_cls": ("dl", "LSTM_cls"),
    }

def get_extra_regression_models():
    return {
        "LR_reg": LinearRegression(),
        "MLP_reg": ("dl", "MLP_reg"),
        "GRU_reg": ("dl", "GRU_reg"),
        "LSTM_reg": ("dl", "LSTM_reg"),
    }

def build_dl_model(input_dim, task, kind):
    model = Sequential()
    if kind in ["GRU","LSTM"]:
        if kind=="GRU":
            model.add(GRU(32, input_shape=(1,input_dim)))
        else:
            model.add(LSTM(32, input_shape=(1,input_dim)))
    else:  # MLP
        model.add(Dense(64, activation="relu", input_shape=(input_dim,)))
        model.add(Dense(32, activation="relu"))

    if task=="classification":
        model.add(Dense(1, activation="sigmoid"))
        model.compile(optimizer=Adam(0.001), loss="binary_crossentropy", metrics=["accuracy"])
    else:
        model.add(Dense(1, activation="linear"))
        model.compile(optimizer=Adam(0.001), loss="mse", metrics=["mae"])
    return model

# === Train new models only ===
all_leader_new = []
all_oos_new    = []

for dataset_tag in DATASET_TAGS:
    if dataset_tag=="base":
        df_src = load_merged_df(BASE_CACHED)
    elif dataset_tag=="eng":
        if ENG_CACHED and Path(ENG_CACHED).exists():
            df_src = load_merged_df(ENG_CACHED)
        else:
            print("[skip] eng missing")
            continue
    else:  # ext
        df_src = load_merged_df(ENG_CACHED) if (ENG_CACHED and Path(ENG_CACHED).exists()) else load_merged_df(BASE_CACHED)

    for asset in ASSETS:
        for freq in FREQS:
            print(f"▶️ Training {asset} [{freq}] on dataset={dataset_tag} (extra models)")
            close_col = ASSET_CLOSE_COLS[asset]
            df_freq = to_frequency(df_src, freq, [close_col])
            dfT     = add_targets_for_asset(df_freq, close_col)
            X, feat_cols = build_X_for_dataset(dfT, close_col, dataset_tag)
            y_cls = dfT["y_true_cls"].values.astype(int)
            y_ret = dfT["y_true_ret"].values.astype(float)
            splits = make_walk_forward_splits(dfT.index, n_folds=(3 if freq=="D" else 2))
            if len(splits)==0:
                print("[warn] Not enough data")
                continue

            # Classification extras
            for name, model in get_extra_classification_models().items():
                prob_all = np.full_like(y_cls, np.nan, dtype=float)
                for k,(tr,va) in enumerate(splits, start=1):
                    Xtr,ytr = X[tr], y_cls[tr]
                    Xva,yva = X[va], y_cls[va]
                    if isinstance(model, tuple) and model[0]=="dl":
                        kind = model[1].split("_")[0]   # "MLP", "GRU", "LSTM"
                        if kind in ["GRU","LSTM"]:
                            Xtr_in = Xtr.reshape((Xtr.shape[0],1,Xtr.shape[1]))
                            Xva_in = Xva.reshape((Xva.shape[0],1,Xva.shape[1]))
                        else:  # MLP
                            Xtr_in, Xva_in = Xtr, Xva
                        m = build_dl_model(Xtr.shape[1], "classification", kind)
                        m.fit(Xtr_in,ytr,epochs=5,batch_size=32,verbose=0)
                        prob = m.predict(Xva_in).flatten()
                    else:
                        model.fit(Xtr,ytr)
                        prob = model.predict_proba(Xva)[:,1]
                    prob_all[va] = prob
                mask = ~np.isnan(prob_all)
                metrics = eval_classification(y_cls[mask], prob_all[mask])
                all_leader_new.append({
                    "asset":asset,"freq":freq,"dataset":dataset_tag,
                    "model":name,"task":"classification",**metrics,
                    "n_samples_oos":int(mask.sum())
                })
                df_oos = pd.DataFrame({
                    "date":dfT.index[mask],"asset":asset,"freq":freq,"dataset":dataset_tag,
                    "model":name,"task":"classification","window":"wf","fold":-1,
                    "y_true_cls":y_cls[mask],"y_true_ret":y_ret[mask],
                    "prob_up":prob_all[mask],"pred_ret":np.nan
                })
                all_oos_new.append(df_oos)

            # Regression extras
            for name, model in get_extra_regression_models().items():
                yhat_all = np.full_like(y_ret, np.nan, dtype=float)
                for k,(tr,va) in enumerate(splits, start=1):
                    Xtr,ytr = X[tr], y_ret[tr]
                    Xva,yva = X[va], y_ret[va]
                    if isinstance(model, tuple) and model[0]=="dl":
                        kind = model[1].split("_")[0]
                        if kind in ["GRU","LSTM"]:
                            Xtr_in = Xtr.reshape((Xtr.shape[0],1,Xtr.shape[1]))
                            Xva_in = Xva.reshape((Xva.shape[0],1,Xva.shape[1]))
                        else:  # MLP
                            Xtr_in, Xva_in = Xtr, Xva
                        m = build_dl_model(Xtr.shape[1], "regression", kind)
                        m.fit(Xtr_in,ytr,epochs=5,batch_size=32,verbose=0)
                        yhat = m.predict(Xva_in).flatten()
                    else:
                        model.fit(Xtr,ytr)
                        yhat = model.predict(Xva)
                    yhat_all[va] = yhat
                mask = ~np.isnan(yhat_all)
                metrics = eval_regression(y_ret[mask], yhat_all[mask])
                all_leader_new.append({
                    "asset":asset,"freq":freq,"dataset":dataset_tag,
                    "model":name,"task":"regression",**metrics,
                    "n_samples_oos":int(mask.sum())
                })
                df_oos = pd.DataFrame({
                    "date":dfT.index[mask],"asset":asset,"freq":freq,"dataset":dataset_tag,
                    "model":name,"task":"regression","window":"wf","fold":-1,
                    "y_true_cls":y_cls[mask],"y_true_ret":y_ret[mask],
                    "prob_up":np.nan,"pred_ret":yhat_all[mask]
                })
                all_oos_new.append(df_oos)

# === Merge with 3.1 and write into 3.2 ===
summary_dir_new = RUN_ROOT_NEW / "60_summary"

predictions_all = pd.concat([preds_31] + all_oos_new, ignore_index=True)
leaderboards_all = pd.concat([leads_31] + [pd.DataFrame(all_leader_new)], ignore_index=True)

predictions_all.to_csv(summary_dir_new / "predictions_master.csv", index=False)
leaderboards_all.to_csv(summary_dir_new / "leaderboards_master.csv", index=False)

coverage = build_coverage(predictions_all)
coverage.to_csv(summary_dir_new / "predictions_coverage.csv", index=False)

signals_latest = build_latest_signals(predictions_all.copy())
signals_latest.to_csv(summary_dir_new / "signals_latest.csv", index=False)

pd.DataFrame(data_dict_rows).to_csv(summary_dir_new / "data_dictionary.csv", index=False)

print("✅ Run 3.2 complete with merged models (RF/XGB from 3.1 + LR/GRU/LSTM/MLP new)")
print("📦 Outputs written to:", summary_dir_new)


RUN_ROOT_NEW: /content/drive/MyDrive/gt-markets/outputs/runs/3.2
Copied cached inputs from 3.1 → 3.2
Loaded 3.1 results: (60000, 12) (96, 15)
▶️ Training BTC [D] on dataset=base (extra models)
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
▶️ Training BTC [W] on dataset=base (extra models)
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step


1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 98ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
▶️ Training GOLD [D] on dataset=base (extra models)
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
▶️ Training GOLD [W] on dataset=base (extra models)


In [6]:
# === Auto-generate README.txt for run 3.2 ===
readme_path = RUN_ROOT_NEW / "README.txt"

readme_text = f"""Run ID: {RUN_ID_NEW}
Notebook: 30_train_predictions.ipynb (with add-on cell for extra models)
Date: 2025-09-23
Mode: FULL RUN (merged with results from 3.1)

--------------------------------------------------------------------
OVERVIEW
--------------------------------------------------------------------
This run ({RUN_ID_NEW}) contains predictions and evaluation metrics for all models:
 - ML: Logistic Regression (LR), Random Forest (RF), XGBoost (XGB)
 - DL: Multi-Layer Perceptron (MLP), GRU, LSTM
For each model we train:
 - Classification task → predict probability the price goes up next period (prob_up)
 - Regression task → predict the size of % change next period (pred_ret)

Walk-forward validation is used:
 - Daily (D): 3 folds (train expands, validate future)
 - Weekly (W): 2 folds
This ensures predictions are always made on unseen future data.

Run {RUN_ID_NEW} is built by:
 - Reading results from run 3.1 (RF + XGB models)
 - Training the missing models (LR, MLP, GRU, LSTM)
 - Combining everything into one summary set

Inputs are cached under 00_meta/input to guarantee reproducibility.

--------------------------------------------------------------------
ASSETS & CLOSE COLUMNS
--------------------------------------------------------------------
BTC     → BTC-USD Close
GOLD    → GC=F Close
OIL     → CL=F Close
USDCNY  → USDCNY=X Close

--------------------------------------------------------------------
DATASETS
--------------------------------------------------------------------
base → Market data only
eng  → Market + engineered features
ext  → Market + engineered + keyword features (using significant lagged keyword correlations)

--------------------------------------------------------------------
SUMMARY FILES (60_summary/)
--------------------------------------------------------------------
1) predictions_master.csv
   - Out-of-sample predictions for all assets × freq × dataset × model × task
   - Columns:
       date, asset, freq, dataset, model, task, window, fold
       y_true_cls, y_true_ret, prob_up, pred_ret
   - y_true_cls → 1 if return > 0, else 0
   - y_true_ret → actual realised % change
   - prob_up    → model probability of upward move (CLS)
   - pred_ret   → model predicted % change (REG)

2) leaderboards_master.csv
   - Evaluation metrics per asset × freq × dataset × model × task
   - Classification: AUC, Accuracy, F1
   - Regression: MAE, RMSE, Spearman, DirHit
   - n_samples_oos = number of out-of-sample rows used

3) predictions_coverage.csv
   - Summary of how many OOS predictions are available for each model
   - Columns: asset, freq, dataset, model, n_pred, start_date, end_date, pct_missing

4) signals_latest.csv
   - Most recent prediction per asset × freq × dataset × model
   - Columns: date, prob_up, pred_ret, is_bigmove_pred, signal_strength
   - is_bigmove_pred → 1 if |pred_ret| exceeds a threshold
   - signal_strength → relative ranking of |pred_ret| (0–1)

5) data_dictionary.csv
   - Column descriptions and examples for reference

--------------------------------------------------------------------
NOTES
--------------------------------------------------------------------
- No backtests are included here. Backtesting and trading logic will be applied
  later in the analysis notebook using predictions_master.csv as input.
- Run 3.1 remains as a reference snapshot with only RF + XGB.
- Run 3.2 should be used as the complete dataset with all models.
- Inputs are cached locally under 00_meta/input for reproducibility.
"""

with open(readme_path, "w") as f:
    f.write(readme_text)

print("✅ README.txt written to:", readme_path)


✅ README.txt written to: /content/drive/MyDrive/gt-markets/outputs/runs/3.2/README.txt
